# 그래프 질의 실습

**SPARQL · Graph Query**

지식 그래프에 저장된 트리플을 패턴으로 찾아오는 질의 방식. 표 형태의 SQL과 달리 관계 경로를 직접 묻는다.

소재 분야에서 이해하기: "이 원소를 포함하고 특정 공정을 거친 시료"를 관계 경로로 한 번에 찾는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [W3C SPARQL 질의 언어](https://www.w3.org/TR/sparql11-query/)

## 1. 아주 작은 그래프 질의 엔진 만들기

변수를 포함한 트리플 패턴을 맞춰보는 것이 그래프 질의의 핵심입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 개념 확인용 작은 지식 그래프. 실제 데이터베이스가 아닙니다.
TRIPLES = [
    # 시료 - 조성
    ('sample:A1', 'hasElement', 'element:Fe'), ('sample:A1', 'hasElement', 'element:Cr'),
    ('sample:A2', 'hasElement', 'element:Fe'), ('sample:A2', 'hasElement', 'element:Ni'),
    ('sample:B1', 'hasElement', 'element:Ti'), ('sample:B1', 'hasElement', 'element:Al'),
    # 시료 - 공정
    ('sample:A1', 'madeBy', 'process:P780'), ('sample:A2', 'madeBy', 'process:P860'),
    ('sample:B1', 'madeBy', 'process:P780'),
    ('process:P780', 'temperatureC', '780'), ('process:P860', 'temperatureC', '860'),
    ('process:P780', 'processType', 'type:Sintering'), ('process:P860', 'processType', 'type:Sintering'),
    # 시료 - 물성
    ('sample:A1', 'hardnessHV', '431'), ('sample:A2', 'hardnessHV', '455'),
    ('sample:B1', 'hardnessHV', '388'),
    # 시료 - 문헌
    ('sample:A1', 'reportedIn', 'paper:10.1000/aaa'), ('sample:A2', 'reportedIn', 'paper:10.1000/aaa'),
    ('sample:B1', 'reportedIn', 'paper:10.1000/bbb'),
    ('paper:10.1000/aaa', 'year', '2026'), ('paper:10.1000/bbb', 'year', '2025'),
]
print('트리플', len(TRIPLES), '개')

In [ ]:
def match(pattern, triples):
    """패턴의 ?로 시작하는 항목을 변수로 보고 바인딩 목록을 돌려줍니다."""
    results = []
    for triple in triples:
        binding = {}
        ok = True
        for slot, value in zip(pattern, triple):
            if slot.startswith('?'):
                if binding.get(slot, value) != value:
                    ok = False; break
                binding[slot] = value
            elif slot != value:
                ok = False; break
        if ok:
            results.append(binding)
    return results

print("패턴 ('?s', 'hardnessHV', '?hv') 결과:")
for binding in match(('?s', 'hardnessHV', '?hv'), TRIPLES):
    print('  ', binding)

## 2. 패턴 여러 개를 이어 붙이기 (조인)

SPARQL의 기본 그래프 패턴은 여러 트리플 패턴을 같은 변수로 잇습니다.

In [ ]:
def query(patterns, triples):
    bindings = [{}]
    for pattern in patterns:
        nxt = []
        for binding in bindings:
            bound = tuple(binding.get(slot, slot) for slot in pattern)
            for extra in match(bound, triples):
                merged = dict(binding)
                merged.update({k: v for k, v in extra.items()})
                nxt.append(merged)
        bindings = nxt
    return bindings

rows = query([('?s', 'hasElement', 'element:Fe'),
              ('?s', 'madeBy', '?p'),
              ('?p', 'temperatureC', '?t'),
              ('?s', 'hardnessHV', '?hv')], TRIPLES)
print('Fe 를 포함한 시료의 공정 온도와 경도:')
for row in rows:
    print('   %-11s %-13s %s C  %s HV' % (row['?s'], row['?p'], row['?t'], row['?hv']))
print('\n같은 질문의 SPARQL 표기:')
print("""SELECT ?s ?t ?hv WHERE {
  ?s hasElement element:Fe .
  ?s madeBy ?p .
  ?p temperatureC ?t .
  ?s hardnessHV ?hv .
}""")

## 3. 표 질의와 무엇이 다른가

같은 데이터를 표로 두면 관계가 늘어날 때마다 열이나 조인이 늘어납니다.

In [ ]:
import pandas as pd

table = pd.DataFrame([
    {'sample': 'A1', 'elements': 'Fe,Cr', 'process': 'P780', 'temp_C': 780, 'hv': 431},
    {'sample': 'A2', 'elements': 'Fe,Ni', 'process': 'P860', 'temp_C': 860, 'hv': 455},
    {'sample': 'B1', 'elements': 'Ti,Al', 'process': 'P780', 'temp_C': 780, 'hv': 388},
])
print(table.to_string(index=False))
print('\n표에서 Fe 포함 시료 찾기:', list(table[table.elements.str.contains('Fe')]['sample']))
print('원소가 문자열로 뭉쳐 있어 "원소 개수", "특정 원소 조합" 질의가 어렵습니다.')
print('\n그래프에서는 원소가 각각 노드이므로 개수도 조합도 바로 물어볼 수 있습니다:')
counts = {}
for binding in match(('?s', 'hasElement', '?e'), TRIPLES):
    counts[binding['?s']] = counts.get(binding['?s'], 0) + 1
print('  시료별 기록된 원소 수:', counts)

## 4. 해석

관계가 자주 바뀌거나 출처가 여러 개일 때 그래프 질의가 유리합니다. 반대로 크기가 크고
형태가 고정된 수치 데이터는 표와 SQL이 더 빠릅니다. 실무에서는 두 방식을 함께 씁니다.
실제로는 `rdflib`나 그래프 데이터베이스가 이 엔진을 대신하고, 여기서는 원리만 확인했습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#sparql)을 여세요.